In [1]:
import sys
print(sys.executable)

c:\CricIQ\venv\Scripts\python.exe


In [4]:

# ------------------------------------------------------------
# CRICKETIQ — NOTEBOOK 08
# STEP 1: LOAD VALIDATED PLAYER FEATURES
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path(r"C:\CricIQ")

PLAYER_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "player_data"
    / "cricketiq_player_match_features.csv"
)

MATCH_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "cricketiq_engineered_features.csv"
)

# ------------------------------------------------------------
# 1. LOAD PLAYER-LEVEL FEATURES
# ------------------------------------------------------------

if not PLAYER_FEATURE_PATH.exists():
    raise FileNotFoundError(PLAYER_FEATURE_PATH)

player_match_features = pd.read_csv(
    PLAYER_FEATURE_PATH,
    low_memory=False
)

# Use the published player-match dataset for coverage auditing.
unified_player_matches = player_match_features.copy()

print("=" * 75)
print("CRICKETIQ — NOTEBOOK 08 DATA LOADING")
print("=" * 75)

print("\nPLAYER FEATURE DATASET")
print("Shape:", player_match_features.shape)
print(
    "Unique matches:",
    player_match_features["match_key"].nunique()
)
print(
    "Unique players:",
    player_match_features["cricketiq_player_id"].nunique()
)

# ------------------------------------------------------------
# 2. LOAD MATCH-LEVEL FEATURES
# ------------------------------------------------------------

if not MATCH_FEATURE_PATH.exists():
    raise FileNotFoundError(MATCH_FEATURE_PATH)

match_features = pd.read_csv(
    MATCH_FEATURE_PATH,
    low_memory=False
)

print("\nMATCH FEATURE DATASET")
print("Shape:", match_features.shape)

# ------------------------------------------------------------
# 3. VERIFY REQUIRED PLAYER COLUMNS
# ------------------------------------------------------------

required_columns = [
    "match_key",
    "match_date",
    "cricketiq_player_id",
    "canonical_player_name",
    "team",
    "opponent",
    "source",
    "has_batting_involvement",
    "has_bowling_record"
]

missing_columns = [
    column for column in required_columns
    if column not in unified_player_matches.columns
]

print("\nREQUIRED COLUMN VALIDATION")

if missing_columns:
    print("Missing columns:", missing_columns)
    raise ValueError("Player dataset validation failed.")

print("All required player columns are available.")

# ------------------------------------------------------------
# 4. VERIFY ENGINEERED FEATURES
# ------------------------------------------------------------

player_pre_match_feature_columns = [
    column for column in player_match_features.columns
    if column.startswith("pre_match_career_")
    or column.startswith("pre_match_recent_")
]

print("\nENGINEERED FEATURE VALIDATION")
print("Pre-match player features:",
      len(player_pre_match_feature_columns))

print("\nDATA LOADING COMPLETED")
print("No existing datasets modified.")


CRICKETIQ — NOTEBOOK 08 DATA LOADING

PLAYER FEATURE DATASET
Shape: (58319, 72)
Unique matches: 2748
Unique players: 2070

MATCH FEATURE DATASET
Shape: (2753, 40)

REQUIRED COLUMN VALIDATION
All required player columns are available.

ENGINEERED FEATURE VALIDATION
Pre-match player features: 46

DATA LOADING COMPLETED
No existing datasets modified.


In [5]:

# ------------------------------------------------------------
# CRICKETIQ — NOTEBOOK 08
# STEP 2: PLAYER-TO-MATCH COVERAGE AUDIT
# ------------------------------------------------------------

print("=" * 75)
print("CRICKETIQ — PLAYER-TO-MATCH COVERAGE AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. PREPARE AUDIT DATA
# ------------------------------------------------------------

coverage_data = unified_player_matches.copy()

coverage_data["both_disciplines"] = (
    coverage_data["has_batting_involvement"].eq(True)
    & coverage_data["has_bowling_record"].eq(True)
)

# ------------------------------------------------------------
# 2. TEAM-MATCH COVERAGE
# ------------------------------------------------------------

team_match_coverage = (
    coverage_data
    .groupby(
        [
            "match_key",
            "match_date",
            "team",
            "opponent",
            "source"
        ],
        dropna=False
    )
    .agg(
        recorded_players=(
            "cricketiq_player_id", "nunique"
        ),
        batting_players=(
            "has_batting_involvement",
            lambda x: x.eq(True).sum()
        ),
        bowling_players=(
            "has_bowling_record",
            lambda x: x.eq(True).sum()
        ),
        players_in_both_disciplines=(
            "both_disciplines", "sum"
        ),
        player_records=(
            "cricketiq_player_id", "size"
        )
    )
    .reset_index()
)

print("\nTEAM-MATCH COVERAGE")
print("Total team-match records:", len(team_match_coverage))

print("\nRecorded players per team-match:")
display(
    team_match_coverage["recorded_players"]
    .describe()
    .to_frame("statistics")
)

# ------------------------------------------------------------
# 3. PLAYER COUNT DISTRIBUTION
# ------------------------------------------------------------

team_coverage_categories = pd.cut(
    team_match_coverage["recorded_players"],
    bins=[-1, 6, 10, 11, float("inf")],
    labels=["0–6 players", "7–10 players", "11 players", "More than 11"]
)

print("\nTEAM-MATCH PLAYER COUNT DISTRIBUTION")

display(
    team_coverage_categories
    .value_counts()
    .sort_index()
    .to_frame("team_match_records")
)

# ------------------------------------------------------------
# 4. MATCH-LEVEL COVERAGE
# ------------------------------------------------------------

match_team_coverage = (
    team_match_coverage
    .groupby("match_key")
    .agg(
        recorded_teams=("team", "nunique"),
        total_recorded_players=("recorded_players", "sum"),
        minimum_players_per_team=("recorded_players", "min"),
        maximum_players_per_team=("recorded_players", "max")
    )
    .reset_index()
)

print("\nMATCH-LEVEL COVERAGE")

display(
    match_team_coverage["recorded_teams"]
    .value_counts()
    .sort_index()
    .to_frame("match_count")
)

print(
    "Matches with exactly two recorded teams:",
    (match_team_coverage["recorded_teams"] == 2).sum()
)

print(
    "Matches with fewer than two recorded teams:",
    (match_team_coverage["recorded_teams"] < 2).sum()
)

print(
    "Matches with more than two recorded teams:",
    (match_team_coverage["recorded_teams"] > 2).sum()
)

# ------------------------------------------------------------
# 5. SOURCE-WISE COVERAGE
# ------------------------------------------------------------

print("\nSOURCE-WISE COVERAGE")

display(
    team_match_coverage
    .groupby("source")
    .agg(
        team_match_records=("match_key", "size"),
        unique_matches=("match_key", "nunique"),
        average_players_per_team_match=("recorded_players", "mean"),
        median_players_per_team_match=("recorded_players", "median"),
        minimum_players=("recorded_players", "min"),
        maximum_players=("recorded_players", "max")
    )
    .round(2)
)

# ------------------------------------------------------------
# 6. LOW-COVERAGE TEAM-MATCH RECORDS
# ------------------------------------------------------------

low_coverage = (
    team_match_coverage[
        team_match_coverage["recorded_players"] < 7
    ]
    .sort_values(["match_date", "team"])
)

print("\nTEAM-MATCH RECORDS WITH FEWER THAN 7 PLAYERS")
print("Count:", len(low_coverage))

display(low_coverage.head(25))

# ------------------------------------------------------------
# 7. DUPLICATE TEAM-MATCH KEY CHECK
# ------------------------------------------------------------

duplicate_team_match_keys = team_match_coverage.duplicated(
    ["match_key", "team"]
).sum()

print("\nDATA INTEGRITY")
print("Duplicate match-team keys:", duplicate_team_match_keys)

print("\nIMPORTANT:")
print(
    "Recorded players are historical scorecard participants, "
    "not necessarily a confirmed playing XI."
)

print("\nSTEP 2 COVERAGE AUDIT COMPLETED — NOT SAVED")


CRICKETIQ — PLAYER-TO-MATCH COVERAGE AUDIT

TEAM-MATCH COVERAGE
Total team-match records: 5496

Recorded players per team-match:


,statistics
count,5496.000000
mean,10.611172
std,1.190811
min,1.000000
25%,11.000000
50%,11.000000
75%,11.000000
max,12.000000



TEAM-MATCH PLAYER COUNT DISTRIBUTION


,team_match_records
recorded_players,
0–6 players,117
7–10 players,849
11 players,4460
More than 11,70



MATCH-LEVEL COVERAGE


,match_count
recorded_teams,
2,2748


Matches with exactly two recorded teams: 2748
Matches with fewer than two recorded teams: 0
Matches with more than two recorded teams: 0

SOURCE-WISE COVERAGE


,team_match_records,unique_matches,average_players_per_team_match,median_players_per_team_match,minimum_players,maximum_players
source,,,,,,
Crickrida,296,148,10.71,11.0,3,11
Cricsheet,5200,2600,10.61,11.0,1,12



TEAM-MATCH RECORDS WITH FEWER THAN 7 PLAYERS
Count: 117


,match_key,match_date,team,opponent,source,recorded_players,batting_players,bowling_players,players_in_both_disciplines,player_records
4680,CS_MATCH_65273,2003-03-04,Pakistan,Zimbabwe,Cricsheet,4,4,0,0,4
4681,CS_MATCH_65273,2003-03-04,Zimbabwe,Pakistan,Cricsheet,3,0,3,0,3
4956,CS_MATCH_66356,2003-04-21,India,South Africa,Cricsheet,5,5,0,0,5
4957,CS_MATCH_66356,2003-04-21,South Africa,India,Cricsheet,4,0,4,0,4
4936,CS_MATCH_66296,2003-07-01,England,Zimbabwe,Cricsheet,6,6,0,0,6
4937,CS_MATCH_66296,2003-07-01,Zimbabwe,England,Cricsheet,6,0,6,0,6
4966,CS_MATCH_66364,2003-10-23,India,New Zealand,Cricsheet,5,5,0,0,5
4516,CS_MATCH_64889,2004-04-24,England,West Indies,Cricsheet,4,0,4,0,4
4517,CS_MATCH_64889,2004-04-24,West Indies,England,Cricsheet,4,4,0,0,4
4983,CS_MATCH_66375,2004-08-23,India,Australia,Cricsheet,5,0,5,0,5



DATA INTEGRITY
Duplicate match-team keys: 0

IMPORTANT:
Recorded players are historical scorecard participants, not necessarily a confirmed playing XI.

STEP 2 COVERAGE AUDIT COMPLETED — NOT SAVED


In [7]:
from pathlib import Path
import pandas as pd

# Path to the published Notebook 07 dataset
player_features_path = Path(
    r"C:\CricIQ\data\features\player_data\cricketiq_player_match_features.csv"
)

# Load the dataset
unified_player_matches = pd.read_csv(
    player_features_path,
    low_memory=False
)

# Recover the exact pre-match feature columns
pre_match_feature_columns = [
    col for col in unified_player_matches.columns
    if col not in [
        "match_key",
        "match_date",
        "team",
        "opponent",
        "source",
        "player_id",
        "player_name",
        "player",
        "batting_team",
        "bowling_team"
    ]
]

print("CRICKETIQ — FEATURE RECOVERY")
print("=" * 60)

print("Dataset shape:", unified_player_matches.shape)
print("Recovered feature columns:", len(pre_match_feature_columns))

print("\nFEATURE COLUMN NAMES:")
for col in pre_match_feature_columns:
    print(col)

print("\nFEATURE RECOVERY COMPLETED")

CRICKETIQ — FEATURE RECOVERY
Dataset shape: (58319, 72)
Recovered feature columns: 67

FEATURE COLUMN NAMES:
cricketiq_player_id
batting_innings_number
runs
balls_faced
fours
sixes
is_out
synthetic_batting_record
dismissal_description
bowling_innings_number
runs_conceded
wickets
legal_balls
wides
no_balls
maidens
canonical_player_name
is_super_over
has_batting_record
has_bowling_record
has_batting_involvement
pre_match_career_matches_played
pre_match_career_batting_innings
pre_match_career_batting_runs
pre_match_career_batting_balls
pre_match_career_batting_fours
pre_match_career_batting_sixes
pre_match_career_batting_dismissals
pre_match_career_bowling_innings
pre_match_career_bowling_runs_conceded
pre_match_career_bowling_wickets
pre_match_career_bowling_legal_balls
pre_match_career_batting_average
pre_match_career_batting_strike_rate
pre_match_career_boundary_runs
pre_match_career_boundary_percentage
pre_match_career_bowling_average
pre_match_career_bowling_economy
pre_match_career_

In [8]:
# Recover ONLY pre-match engineered features

pre_match_feature_columns = [
    col for col in unified_player_matches.columns
    if col.startswith("pre_match_")
]

print("=" * 65)
print("CRICKETIQ — PRE-MATCH FEATURE VALIDATION")
print("=" * 65)

print("Total recovered pre-match features:",
      len(pre_match_feature_columns))

print("\nFEATURE LIST:")

for i, col in enumerate(pre_match_feature_columns, start=1):
    print(f"{i:02d}. {col}")

print("\nVALIDATION")

if len(pre_match_feature_columns) == 46:
    print("SUCCESS: All 46 pre-match features recovered.")
else:
    print("WARNING: Expected 46 features, found:",
          len(pre_match_feature_columns))

CRICKETIQ — PRE-MATCH FEATURE VALIDATION
Total recovered pre-match features: 46

FEATURE LIST:
01. pre_match_career_matches_played
02. pre_match_career_batting_innings
03. pre_match_career_batting_runs
04. pre_match_career_batting_balls
05. pre_match_career_batting_fours
06. pre_match_career_batting_sixes
07. pre_match_career_batting_dismissals
08. pre_match_career_bowling_innings
09. pre_match_career_bowling_runs_conceded
10. pre_match_career_bowling_wickets
11. pre_match_career_bowling_legal_balls
12. pre_match_career_batting_average
13. pre_match_career_batting_strike_rate
14. pre_match_career_boundary_runs
15. pre_match_career_boundary_percentage
16. pre_match_career_bowling_average
17. pre_match_career_bowling_economy
18. pre_match_career_bowling_strike_rate
19. pre_match_recent_5_batting_innings
20. pre_match_recent_5_batting_runs
21. pre_match_recent_5_batting_balls
22. pre_match_recent_5_batting_dismissals
23. pre_match_recent_5_batting_average
24. pre_match_recent_5_batting_st

In [9]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 3B: TEAM-LEVEL PLAYER FEATURE AGGREGATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("CRICKETIQ — TEAM-LEVEL PLAYER FEATURE AGGREGATION")
print("=" * 75)

team_player_data = unified_player_matches.copy()

# Team-match identification columns
group_columns = [
    "match_key",
    "match_date",
    "team",
    "opponent",
    "source"
]

# Rate/statistical features: do not sum these
rate_keywords = [
    "average",
    "strike_rate",
    "economy",
    "percentage"
]

rate_features = [
    col for col in pre_match_feature_columns
    if any(keyword in col.lower() for keyword in rate_keywords)
]

# Count and volume features
volume_features = [
    col for col in pre_match_feature_columns
    if col not in rate_features
]

print("\nFEATURE CLASSIFICATION")
print("Rate features:", len(rate_features))
print("Volume/count features:", len(volume_features))

# ------------------------------------------------------------
# 1. Build aggregation instructions
# ------------------------------------------------------------

aggregation = {}

for col in rate_features:
    aggregation[col] = ["mean", "max"]

for col in volume_features:
    aggregation[col] = ["mean", "max", "sum"]

# ------------------------------------------------------------
# 2. Aggregate player features
# ------------------------------------------------------------

team_player_features = (
    team_player_data
    .groupby(group_columns, dropna=False)
    .agg(aggregation)
)

# Flatten column names
team_player_features.columns = [
    f"{feature}_{stat}"
    for feature, stat in team_player_features.columns
]

team_player_features = team_player_features.reset_index()

# ------------------------------------------------------------
# 3. Add player coverage information
# ------------------------------------------------------------

team_player_counts = (
    team_player_data
    .groupby(group_columns, dropna=False)
    .agg(
        recorded_players=("cricketiq_player_id", "nunique"),
        player_records=("cricketiq_player_id", "size")
    )
    .reset_index()
)

team_player_features = team_player_features.merge(
    team_player_counts,
    on=group_columns,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 4. Add coverage indicators
# ------------------------------------------------------------

team_player_features["has_player_data"] = (
    team_player_features["recorded_players"] > 0
)

team_player_features["low_player_coverage"] = (
    team_player_features["recorded_players"] < 7
)

team_player_features["full_player_coverage"] = (
    team_player_features["recorded_players"] == 11
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\nTEAM-LEVEL FEATURE DATASET")
print("Shape:", team_player_features.shape)

print("\nSAMPLE RECORDS")
display(team_player_features.head())

print("\nPLAYER COVERAGE SUMMARY")
display(
    team_player_features["recorded_players"].describe()
)

print("\nFEATURE AGGREGATION COMPLETED")

CRICKETIQ — TEAM-LEVEL PLAYER FEATURE AGGREGATION

FEATURE CLASSIFICATION
Rate features: 18
Volume/count features: 28

TEAM-LEVEL FEATURE DATASET
Shape: (5496, 130)

SAMPLE RECORDS


,match_key,match_date,team,opponent,source,pre_match_career_batting_average_mean,pre_match_career_batting_average_max,pre_match_career_batting_strike_rate_mean,pre_match_career_batting_strike_rate_max,pre_match_career_boundary_percentage_mean,...,pre_match_recent_10_bowling_legal_balls_max,pre_match_recent_10_bowling_legal_balls_sum,pre_match_recent_10_bowling_wickets_mean,pre_match_recent_10_bowling_wickets_max,pre_match_recent_10_bowling_wickets_sum,recorded_players,player_records,has_player_data,low_player_coverage,full_player_coverage
0,CR_MATCH_20100703_Afghanistan_Ireland,2010-07-03,Afghanistan,Ireland,Crickrida,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.000000,0.0,0.0,11,11,True,False,True
1,CR_MATCH_20100703_Afghanistan_Ireland,2010-07-03,Ireland,Afghanistan,Crickrida,23.388167,35.000000,71.221491,85.365854,47.649108,...,416.0,1209.0,3.181818,11.0,35.0,11,11,True,False,True
2,CR_MATCH_20120210_Afghanistan_Pakistan,2012-02-10,Afghanistan,Pakistan,Crickrida,14.166667,25.000000,62.093741,96.153846,26.905176,...,60.0,204.0,0.272727,1.0,3.0,11,11,True,False,True
3,CR_MATCH_20120210_Afghanistan_Pakistan,2012-02-10,Pakistan,Afghanistan,Crickrida,24.715074,41.283019,78.106370,135.004965,43.481407,...,528.0,2864.0,8.600000,18.0,86.0,10,10,True,False,False
4,CR_MATCH_20120705_Afghanistan_Ireland,2012-07-05,Afghanistan,Ireland,Crickrida,16.937500,32.500000,58.733589,104.761905,39.858982,...,108.0,373.0,0.545455,2.0,6.0,11,11,True,False,True



PLAYER COVERAGE SUMMARY


count    5496.000000
mean       10.611172
std         1.190811
min         1.000000
25%        11.000000
50%        11.000000
75%        11.000000
max        12.000000
Name: recorded_players, dtype: float64


FEATURE AGGREGATION COMPLETED


In [10]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 4: TEAM-LEVEL AGGREGATION QUALITY AUDIT
# ============================================================

print("=" * 75)
print("CRICKETIQ — TEAM-LEVEL AGGREGATION QUALITY AUDIT")
print("=" * 75)

# 1. Duplicate team-match keys
duplicate_count = team_player_features.duplicated(
    subset=group_columns
).sum()

print("\n1. DUPLICATE CHECK")
print("Duplicate team-match records:", duplicate_count)

# 2. Match-level team coverage
teams_per_match = (
    team_player_features
    .groupby("match_key")["team"]
    .nunique()
)

print("\n2. MATCH-LEVEL COVERAGE")
print("Unique matches:", len(teams_per_match))
print("Matches with exactly 2 teams:",
      (teams_per_match == 2).sum())
print("Matches with fewer than 2 teams:",
      (teams_per_match < 2).sum())
print("Matches with more than 2 teams:",
      (teams_per_match > 2).sum())

# 3. Missing values
feature_columns = [
    col for col in team_player_features.columns
    if col not in group_columns
]

missing_summary = (
    team_player_features[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n3. MISSING VALUE AUDIT")
print("Total missing feature cells:",
      missing_summary.sum())

print("\nTop 15 columns with missing values:")
display(missing_summary.head(15))

# 4. Infinite values
numeric_data = team_player_features.select_dtypes(
    include=np.number
)

infinite_count = np.isinf(numeric_data).sum().sum()

print("\n4. INFINITE VALUE CHECK")
print("Infinite values:", infinite_count)

# 5. Original feature representation
represented_features = [
    col for col in pre_match_feature_columns
    if any(
        col + "_" + stat in team_player_features.columns
        for stat in ["mean", "max", "sum"]
    )
]

print("\n5. FEATURE REPRESENTATION")
print("Original pre-match features:", len(pre_match_feature_columns))
print("Features represented after aggregation:",
      len(represented_features))
print("Unrepresented features:",
      len(pre_match_feature_columns) - len(represented_features))

# 6. Final summary
print("\n" + "=" * 75)
print("AGGREGATION QUALITY AUDIT COMPLETED")
print("=" * 75)

CRICKETIQ — TEAM-LEVEL AGGREGATION QUALITY AUDIT

1. DUPLICATE CHECK
Duplicate team-match records: 0

2. MATCH-LEVEL COVERAGE
Unique matches: 2748
Matches with exactly 2 teams: 2748
Matches with fewer than 2 teams: 0
Matches with more than 2 teams: 0

3. MISSING VALUE AUDIT
Total missing feature cells: 1044

Top 15 columns with missing values:


pre_match_recent_5_bowling_average_max          34
pre_match_recent_5_bowling_strike_rate_mean     34
pre_match_recent_5_bowling_average_mean         34
pre_match_recent_5_bowling_strike_rate_max      34
pre_match_recent_10_bowling_average_mean        33
pre_match_recent_10_bowling_strike_rate_mean    33
pre_match_recent_10_bowling_average_max         33
pre_match_recent_10_bowling_strike_rate_max     33
pre_match_career_bowling_strike_rate_mean       32
pre_match_career_bowling_average_max            32
pre_match_career_bowling_strike_rate_max        32
pre_match_career_bowling_average_mean           32
pre_match_career_bowling_economy_mean           30
pre_match_recent_5_bowling_economy_mean         30
pre_match_recent_10_bowling_economy_max         30
dtype: int64


4. INFINITE VALUE CHECK
Infinite values: 0

5. FEATURE REPRESENTATION
Original pre-match features: 46
Features represented after aggregation: 46
Unrepresented features: 0

AGGREGATION QUALITY AUDIT COMPLETED


In [11]:
# ============================================================
# CRICKETIQ — STEP 4A
# MISSING VALUE DIAGNOSTICS
# ============================================================

print("=" * 75)
print("CRICKETIQ — MISSING VALUE DIAGNOSTICS")
print("=" * 75)

# 1. Identify aggregated columns by feature type
rate_agg_columns = [
    f"{col}_{stat}"
    for col in rate_features
    for stat in ["mean", "max"]
]

volume_agg_columns = [
    f"{col}_{stat}"
    for col in volume_features
    for stat in ["mean", "max", "sum"]
]

# 2. Count missing values by category
rate_missing = (
    team_player_features[rate_agg_columns]
    .isna().sum().sum()
)

volume_missing = (
    team_player_features[volume_agg_columns]
    .isna().sum().sum()
)

print("\n1. MISSING VALUES BY FEATURE CATEGORY")
print("Rate feature missing cells:", rate_missing)
print("Volume/count missing cells:", volume_missing)
print("Total missing cells:", rate_missing + volume_missing)

# 3. Find team-match records with missing aggregated values
missing_mask = (
    team_player_features[
        rate_agg_columns + volume_agg_columns
    ].isna().any(axis=1)
)

affected_records = team_player_features.loc[
    missing_mask,
    group_columns + ["recorded_players"]
].copy()

print("\n2. AFFECTED TEAM-MATCH RECORDS")
print("Records with at least one missing value:",
      len(affected_records))

display(affected_records.head(15))

# 4. Show the columns with the most missing values
missing_by_column = (
    team_player_features[
        rate_agg_columns + volume_agg_columns
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n3. MISSING VALUES BY AGGREGATED FEATURE")
display(missing_by_column[missing_by_column > 0])

# 5. Check missing values in original player-level features
player_missing = (
    team_player_data[pre_match_feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n4. PLAYER-LEVEL MISSING VALUES")
display(player_missing[player_missing > 0].head(20))

print("\nMISSING VALUE DIAGNOSTICS COMPLETED")

CRICKETIQ — MISSING VALUE DIAGNOSTICS

1. MISSING VALUES BY FEATURE CATEGORY
Rate feature missing cells: 1044
Volume/count missing cells: 0
Total missing cells: 1044

2. AFFECTED TEAM-MATCH RECORDS
Records with at least one missing value: 34


,match_key,match_date,team,opponent,source,recorded_players
0,CR_MATCH_20100703_Afghanistan_Ireland,2010-07-03,Afghanistan,Ireland,Crickrida,11
373,CS_MATCH_1026847,2016-11-04,Papua New Guinea,Hong Kong,Cricsheet,11
412,CS_MATCH_1031669,2017-09-21,England,West Indies,Cricsheet,2
680,CS_MATCH_1140379,2018-10-10,England,Sri Lanka,Cricsheet,4
886,CS_MATCH_1154649,2018-08-30,Nepal,United Arab Emirates,Cricsheet,11
961,CS_MATCH_1174844,2019-04-27,United States of America,Papua New Guinea,Cricsheet,11
962,CS_MATCH_1174845,2019-04-27,Namibia,Oman,Cricsheet,11
963,CS_MATCH_1174845,2019-04-27,Oman,Namibia,Cricsheet,11
1472,CS_MATCH_1322279,2022-11-27,India,New Zealand,Cricsheet,3
1678,CS_MATCH_1365097,2023-03-30,Jersey,Namibia,Cricsheet,11



3. MISSING VALUES BY AGGREGATED FEATURE


pre_match_recent_5_bowling_strike_rate_mean     34
pre_match_recent_5_bowling_strike_rate_max      34
pre_match_recent_5_bowling_average_max          34
pre_match_recent_5_bowling_average_mean         34
pre_match_recent_10_bowling_strike_rate_max     33
pre_match_recent_10_bowling_strike_rate_mean    33
pre_match_recent_10_bowling_average_max         33
pre_match_recent_10_bowling_average_mean        33
pre_match_career_bowling_average_mean           32
pre_match_career_bowling_average_max            32
pre_match_career_bowling_strike_rate_mean       32
pre_match_career_bowling_strike_rate_max        32
pre_match_recent_10_bowling_economy_mean        30
pre_match_recent_10_bowling_economy_max         30
pre_match_recent_5_bowling_economy_max          30
pre_match_recent_5_bowling_economy_mean         30
pre_match_career_bowling_economy_mean           30
pre_match_career_bowling_economy_max            30
pre_match_career_batting_strike_rate_mean       26
pre_match_career_batting_strike


4. PLAYER-LEVEL MISSING VALUES


pre_match_recent_5_bowling_average         22036
pre_match_recent_5_bowling_strike_rate     22036
pre_match_recent_10_bowling_average        20682
pre_match_recent_10_bowling_strike_rate    20682
pre_match_career_bowling_strike_rate       20295
pre_match_career_bowling_average           20295
pre_match_recent_10_bowling_economy        15587
pre_match_career_bowling_economy           15587
pre_match_recent_5_bowling_economy         15587
pre_match_recent_5_batting_average          3980
pre_match_career_batting_average            3894
pre_match_recent_10_batting_average         3894
pre_match_recent_5_boundary_percentage      3360
pre_match_career_boundary_percentage        3342
pre_match_recent_10_boundary_percentage     3342
pre_match_career_batting_strike_rate        2859
pre_match_recent_10_batting_strike_rate     2859
pre_match_recent_5_batting_strike_rate      2859
dtype: int64


MISSING VALUE DIAGNOSTICS COMPLETED


In [12]:
# ============================================================
# CRICKETIQ — STEP 4B
# INSPECT MISSING RATE EXAMPLES
# ============================================================

print("=" * 75)
print("CRICKETIQ — MISSING RATE EXAMPLE INVESTIGATION")
print("=" * 75)

# Select important missing-rate columns
inspection_columns = [
    "pre_match_career_batting_average",
    "pre_match_career_bowling_average",
    "pre_match_career_bowling_economy",
    "pre_match_career_batting_strike_rate",
    "pre_match_career_bowling_strike_rate"
]

# Original player-level records for the Afghanistan example
example_players = team_player_data[
    (team_player_data["match_key"] ==
     "CR_MATCH_20100703_Afghanistan_Ireland")
    &
    (team_player_data["team"] == "Afghanistan")
].copy()

print("\n1. AFGHANISTAN — 2010-07-03")

print("Recorded players:", len(example_players))

display(
    example_players[
        ["canonical_player_name"] + inspection_columns
    ]
)

# Count missing values within this team
print("\n2. MISSING VALUES WITHIN THIS TEAM")

display(
    example_players[inspection_columns]
    .isna()
    .sum()
    .to_frame("missing_player_values")
)

# Inspect all affected team-match records
print("\n3. SUMMARY OF ALL AFFECTED RECORDS")

display(
    affected_records[
        ["match_key", "team", "recorded_players"]
    ].head(20)
)

print("\nINVESTIGATION COMPLETED")

CRICKETIQ — MISSING RATE EXAMPLE INVESTIGATION

1. AFGHANISTAN — 2010-07-03
Recorded players: 11


,canonical_player_name,pre_match_career_batting_average,pre_match_career_bowling_average,pre_match_career_bowling_economy,pre_match_career_batting_strike_rate,pre_match_career_bowling_strike_rate
17015,Mohammad Shahzad,NaN,NaN,NaN,NaN,NaN
17025,Asghar Afghan,NaN,NaN,NaN,NaN,NaN
17026,Hamid Hassan,NaN,NaN,NaN,NaN,NaN
17028,Karim Sadiq,NaN,NaN,NaN,NaN,NaN
17029,Khaliq Dad,NaN,NaN,NaN,NaN,NaN
17030,Mohammad Nabi,NaN,NaN,NaN,NaN,NaN
17031,Nawroz Mangal,NaN,NaN,NaN,NaN,NaN
17032,Noor Ali Zadran,NaN,NaN,NaN,NaN,NaN
17033,Samiullah Shinwari,NaN,NaN,NaN,NaN,NaN
17034,Shabir Noori,NaN,NaN,NaN,NaN,NaN



2. MISSING VALUES WITHIN THIS TEAM


,missing_player_values
pre_match_career_batting_average,11
pre_match_career_bowling_average,11
pre_match_career_bowling_economy,11
pre_match_career_batting_strike_rate,11
pre_match_career_bowling_strike_rate,11



3. SUMMARY OF ALL AFFECTED RECORDS


,match_key,team,recorded_players
0,CR_MATCH_20100703_Afghanistan_Ireland,Afghanistan,11
373,CS_MATCH_1026847,Papua New Guinea,11
412,CS_MATCH_1031669,England,2
680,CS_MATCH_1140379,England,4
886,CS_MATCH_1154649,Nepal,11
961,CS_MATCH_1174844,United States of America,11
962,CS_MATCH_1174845,Namibia,11
963,CS_MATCH_1174845,Oman,11
1472,CS_MATCH_1322279,India,3
1678,CS_MATCH_1365097,Jersey,11



INVESTIGATION COMPLETED


In [13]:
# ============================================================
# CRICKETIQ — STEP 4C
# HISTORICAL EXPERIENCE AVAILABILITY AUDIT
# ============================================================

print("=" * 75)
print("CRICKETIQ — HISTORICAL EXPERIENCE AVAILABILITY AUDIT")
print("=" * 75)

# Select the Afghanistan example
afghanistan_players = team_player_data[
    (team_player_data["match_key"] ==
     "CR_MATCH_20100703_Afghanistan_Ireland")
    &
    (team_player_data["team"] == "Afghanistan")
].copy()

experience_columns = [
    "canonical_player_name",
    "pre_match_career_matches_played",
    "pre_match_career_batting_innings",
    "pre_match_career_batting_runs",
    "pre_match_career_bowling_innings",
    "pre_match_career_bowling_wickets",
    "pre_match_career_batting_average",
    "pre_match_career_bowling_average"
]

print("\n1. AFGHANISTAN PLAYER EXPERIENCE")

display(
    afghanistan_players[experience_columns]
)

# Summarize prior experience
print("\n2. EXPERIENCE SUMMARY")

display(
    afghanistan_players[
        experience_columns[1:]
    ].describe().T
)

# Count players with no recorded prior match history
print("\n3. PRIOR MATCH HISTORY")

print(
    "Players with zero prior recorded matches:",
    (
        afghanistan_players[
            "pre_match_career_matches_played"
        ] == 0
    ).sum()
)

print(
    "Players with missing prior match count:",
    afghanistan_players[
        "pre_match_career_matches_played"
    ].isna().sum()
)

print("\nHISTORICAL EXPERIENCE AUDIT COMPLETED")

CRICKETIQ — HISTORICAL EXPERIENCE AVAILABILITY AUDIT

1. AFGHANISTAN PLAYER EXPERIENCE


,canonical_player_name,pre_match_career_matches_played,pre_match_career_batting_innings,pre_match_career_batting_runs,pre_match_career_bowling_innings,pre_match_career_bowling_wickets,pre_match_career_batting_average,pre_match_career_bowling_average
17015,Mohammad Shahzad,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17025,Asghar Afghan,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17026,Hamid Hassan,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17028,Karim Sadiq,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17029,Khaliq Dad,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17030,Mohammad Nabi,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17031,Nawroz Mangal,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17032,Noor Ali Zadran,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17033,Samiullah Shinwari,0.0,0.0,0.0,0.0,0.0,NaN,NaN
17034,Shabir Noori,0.0,0.0,0.0,0.0,0.0,NaN,NaN



2. EXPERIENCE SUMMARY


,count,mean,std,min,25%,50%,75%,max
pre_match_career_matches_played,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pre_match_career_batting_innings,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pre_match_career_batting_runs,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pre_match_career_bowling_innings,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pre_match_career_bowling_wickets,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pre_match_career_batting_average,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pre_match_career_bowling_average,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



3. PRIOR MATCH HISTORY
Players with zero prior recorded matches: 11
Players with missing prior match count: 0

HISTORICAL EXPERIENCE AUDIT COMPLETED


In [14]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 5: FINAL TEAM-LEVEL DATASET VALIDATION
# ============================================================

print("=" * 75)
print("CRICKETIQ — FINAL TEAM-LEVEL DATASET VALIDATION")
print("=" * 75)

# Work on a final copy
final_team_features = team_player_features.copy()

# Add player coverage ratio
final_team_features["player_coverage_ratio"] = (
    final_team_features["recorded_players"] / 11
)

# Validate uniqueness
duplicate_keys = final_team_features.duplicated(
    subset=group_columns
).sum()

# Validate match coverage
team_counts = (
    final_team_features.groupby("match_key")["team"].nunique()
)

# Validate feature columns
aggregated_features = [
    col for col in final_team_features.columns
    if col not in group_columns
    + [
        "recorded_players",
        "player_records",
        "has_player_data",
        "low_player_coverage",
        "full_player_coverage",
        "player_coverage_ratio"
    ]
]

# Validate numeric values
numeric_values = final_team_features.select_dtypes(
    include=np.number
)

infinite_values = np.isinf(numeric_values).sum().sum()

# Display summary
print("\nFINAL DATASET SUMMARY")
print("Shape:", final_team_features.shape)
print("Unique matches:", final_team_features["match_key"].nunique())
print("Team-match records:", len(final_team_features))
print("Aggregated player features:", len(aggregated_features))

print("\nDATA INTEGRITY")
print("Duplicate team-match keys:", duplicate_keys)
print("Matches with exactly 2 teams:", (team_counts == 2).sum())
print("Infinite values:", infinite_values)

print("\nCOVERAGE SUMMARY")
print("Low coverage records:",
      final_team_features["low_player_coverage"].sum())

print("Full coverage records:",
      final_team_features["full_player_coverage"].sum())

print("Average coverage ratio:",
      round(final_team_features["player_coverage_ratio"].mean(), 4))

print("\nMISSING VALUE SUMMARY")
print("Missing values:",
      final_team_features[aggregated_features].isna().sum().sum())

print("\nFINAL DATASET VALIDATION COMPLETED")

CRICKETIQ — FINAL TEAM-LEVEL DATASET VALIDATION

FINAL DATASET SUMMARY
Shape: (5496, 131)
Unique matches: 2748
Team-match records: 5496
Aggregated player features: 120

DATA INTEGRITY
Duplicate team-match keys: 0
Matches with exactly 2 teams: 2748
Infinite values: 0

COVERAGE SUMMARY
Low coverage records: 117
Full coverage records: 4460
Average coverage ratio: 0.9647

MISSING VALUE SUMMARY
Missing values: 1044

FINAL DATASET VALIDATION COMPLETED


In [15]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 6: SAVE TEAM-LEVEL PLAYER FEATURES
# ============================================================

from pathlib import Path
import pandas as pd

# Output directory
output_dir = Path(
    r"C:\CricIQ\data\features\player_data"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Output file
output_path = (
    output_dir / "cricketiq_team_match_player_features.csv"
)

# Save dataset
final_team_features.to_csv(
    output_path,
    index=False
)

# Reload to verify
saved_team_features = pd.read_csv(
    output_path,
    low_memory=False
)

print("=" * 75)
print("CRICKETIQ — TEAM-LEVEL DATASET PUBLICATION")
print("=" * 75)

print("\nSaved path:")
print(output_path)

print("\nOriginal shape:", final_team_features.shape)
print("Reloaded shape:", saved_team_features.shape)

print(
    "\nDuplicate team-match keys:",
    saved_team_features.duplicated(
        subset=group_columns
    ).sum()
)

print(
    "Missing values:",
    saved_team_features.isna().sum().sum()
)

if (
    final_team_features.shape == saved_team_features.shape
    and saved_team_features.duplicated(
        subset=group_columns
    ).sum() == 0
):
    print("\nDATASET SAVED AND RELOADED SUCCESSFULLY")
else:
    print("\nWARNING: PLEASE INVESTIGATE DATASET")

CRICKETIQ — TEAM-LEVEL DATASET PUBLICATION

Saved path:
C:\CricIQ\data\features\player_data\cricketiq_team_match_player_features.csv

Original shape: (5496, 131)
Reloaded shape: (5496, 131)

Duplicate team-match keys: 0
Missing values: 1044

DATASET SAVED AND RELOADED SUCCESSFULLY


In [17]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 7A: INSPECT NOTEBOOK 05 DATASET STRUCTURE
# ============================================================

print("=" * 75)
print("CRICKETIQ — MATCH DATASET STRUCTURE INSPECTION")
print("=" * 75)

print("\nDATASET SHAPE:")
print(match_features.shape)

print("\nALL COLUMN NAMES:")

for i, col in enumerate(match_features.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nPOSSIBLE MATCH IDENTIFIER COLUMNS:")

for col in match_features.columns:
    if any(word in col.lower() for word in
           ["match", "id", "date", "team", "source"]):
        print(col)

print("\nFIRST FIVE MATCH RECORDS:")
display(match_features.head())

print("\nTEAM-LEVEL DATASET IDENTIFIERS:")
print(team_features.columns[:10].tolist())

print("\nINSPECTION COMPLETED")

CRICKETIQ — MATCH DATASET STRUCTURE INSPECTION

DATASET SHAPE:
(2753, 40)

ALL COLUMN NAMES:
01. match_date
02. team_1
03. team_2
04. venue
05. city
06. toss_winner
07. toss_decision
08. winner
09. result_type
10. source
11. match_id
12. team_1_win
13. team_1_win_pct
14. team_2_win_pct
15. win_pct_difference
16. team_1_recent_form
17. team_2_recent_form
18. recent_form_difference
19. team_1_h2h_win_pct
20. team_2_h2h_win_pct
21. h2h_win_pct_difference
22. team_1_venue_win_pct
23. team_2_venue_win_pct
24. venue_win_pct_difference
25. city_original
26. city_standard
27. venue_country
28. team_1_is_home
29. team_2_is_home
30. neutral_venue
31. home_advantage
32. team_1_home_win_pct
33. team_2_home_win_pct
34. team_1_away_win_pct
35. team_2_away_win_pct
36. home_win_pct_difference
37. away_win_pct_difference
38. team_1_elo
39. team_2_elo
40. elo_difference

POSSIBLE MATCH IDENTIFIER COLUMNS:
match_date
team_1
team_2
source
match_id
team_1_win
team_1_win_pct
team_2_win_pct
team_1_recent_for

,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,...,home_advantage,team_1_home_win_pct,team_2_home_win_pct,team_1_away_win_pct,team_2_away_win_pct,home_win_pct_difference,away_win_pct_difference,team_1_elo,team_2_elo,elo_difference
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",Nottingham,England,bat,England,Win,Cricsheet,...,1,0.5,0.5,0.500000,0.5,0.0,0.000000,1500.000000,1500.000000,0.000000
1,2002-06-29,England,India,"Lord's, London",London,England,bat,India,Win,Cricsheet,...,1,1.0,0.5,0.500000,0.5,0.5,0.000000,1516.000000,1500.000000,16.000000
2,2002-12-29,New Zealand,India,"McLean Park, Napier",Napier,India,field,New Zealand,Win,Cricsheet,...,1,0.5,0.5,0.500000,1.0,0.0,-0.500000,1500.000000,1516.736307,-16.736307
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",Christchurch,India,bat,New Zealand,Win,Cricsheet,...,-1,0.5,1.0,0.500000,0.5,-0.5,0.000000,1499.966167,1516.770140,-16.803973
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",Queenstown,New Zealand,field,New Zealand,Win,Cricsheet,...,-1,0.5,1.0,0.333333,0.5,-0.5,-0.166667,1484.739416,1531.996891,-47.257475



TEAM-LEVEL DATASET IDENTIFIERS:
['match_key', 'match_date', 'team', 'opponent', 'source', 'pre_match_career_batting_average_mean', 'pre_match_career_batting_average_max', 'pre_match_career_batting_strike_rate_mean', 'pre_match_career_batting_strike_rate_max', 'pre_match_career_boundary_percentage_mean']

INSPECTION COMPLETED


In [18]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 7B: CREATE COMMON MATCH IDENTIFIERS
# ============================================================

# Work on independent copies
match_integration = match_features.copy()
team_integration = team_features.copy()

# Standardize dates
match_integration["match_date"] = pd.to_datetime(
    match_integration["match_date"]
).dt.strftime("%Y-%m-%d")

team_integration["match_date"] = pd.to_datetime(
    team_integration["match_date"]
).dt.strftime("%Y-%m-%d")

# Normalize team names and source labels
def normalize_text(series):
    return series.astype("string").str.strip().str.casefold()

# Create order-independent match key
def create_match_key(df, team_a, team_b):
    first = normalize_text(df[team_a])
    second = normalize_text(df[team_b])

    return (
        df["match_date"].astype(str)
        + "|"
        + df["source"].astype("string").str.strip().str.casefold()
        + "|"
        + np.minimum(first, second).astype(str)
        + "|"
        + np.maximum(first, second).astype(str)
    )

# Match-level key
match_integration["integration_key"] = create_match_key(
    match_integration,
    "team_1",
    "team_2"
)

# Team-level key
team_integration["integration_key"] = create_match_key(
    team_integration,
    "team",
    "opponent"
)

print("=" * 75)
print("CRICKETIQ — COMMON MATCH KEY VALIDATION")
print("=" * 75)

print("\nMATCH DATASET")
print("Rows:", len(match_integration))
print("Unique integration keys:",
      match_integration["integration_key"].nunique())
print("Duplicate keys:",
      match_integration["integration_key"].duplicated().sum())

print("\nTEAM DATASET")
print("Rows:", len(team_integration))
print("Unique integration keys:",
      team_integration["integration_key"].nunique())

print("\nSAMPLE MATCH KEYS")
display(
    match_integration[
        ["match_date", "team_1", "team_2",
         "source", "integration_key"]
    ].head()
)

print("\nSAMPLE TEAM KEYS")
display(
    team_integration[
        ["match_date", "team", "opponent",
         "source", "integration_key"]
    ].head()
)

print("\nCOMMON MATCH KEY CREATION COMPLETED")

CRICKETIQ — COMMON MATCH KEY VALIDATION

MATCH DATASET
Rows: 2753
Unique integration keys: 2753
Duplicate keys: 0

TEAM DATASET
Rows: 5496
Unique integration keys: 2748

SAMPLE MATCH KEYS


,match_date,team_1,team_2,source,integration_key
0,2002-06-27,England,Sri Lanka,Cricsheet,2002-06-27|cricsheet|england|sri lanka
1,2002-06-29,England,India,Cricsheet,2002-06-29|cricsheet|england|india
2,2002-12-29,New Zealand,India,Cricsheet,2002-12-29|cricsheet|india|new zealand
3,2003-01-01,India,New Zealand,Cricsheet,2003-01-01|cricsheet|india|new zealand
4,2003-01-04,India,New Zealand,Cricsheet,2003-01-04|cricsheet|india|new zealand



SAMPLE TEAM KEYS


,match_date,team,opponent,source,integration_key
0,2010-07-03,Afghanistan,Ireland,Crickrida,2010-07-03|crickrida|afghanistan|ireland
1,2010-07-03,Ireland,Afghanistan,Crickrida,2010-07-03|crickrida|afghanistan|ireland
2,2012-02-10,Afghanistan,Pakistan,Crickrida,2012-02-10|crickrida|afghanistan|pakistan
3,2012-02-10,Pakistan,Afghanistan,Crickrida,2012-02-10|crickrida|afghanistan|pakistan
4,2012-07-05,Afghanistan,Ireland,Crickrida,2012-07-05|crickrida|afghanistan|ireland



COMMON MATCH KEY CREATION COMPLETED


In [19]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 7C: MATCH-KEY COVERAGE VALIDATION
# ============================================================

print("=" * 75)
print("CRICKETIQ — MATCH-KEY COVERAGE VALIDATION")
print("=" * 75)

# Unique keys from both datasets
match_keys = set(match_integration["integration_key"])
player_keys = set(team_integration["integration_key"])

# Compare key coverage
matched_keys = match_keys.intersection(player_keys)
unmatched_match_keys = match_keys - player_keys
orphan_player_keys = player_keys - match_keys

# Count player team records per match
player_team_counts = (
    team_integration
    .groupby("integration_key")
    .size()
)

print("\n1. MATCH KEY COVERAGE")
print("Total match dataset keys:", len(match_keys))
print("Total player dataset keys:", len(player_keys))
print("Matching keys:", len(matched_keys))
print("Matches without player data:", len(unmatched_match_keys))
print("Player keys absent from match dataset:", len(orphan_player_keys))

# Check two team records per player match
print("\n2. PLAYER TEAM-RECORD VALIDATION")

print(
    "Matches with exactly 2 team records:",
    (player_team_counts == 2).sum()
)

print(
    "Matches with fewer than 2 team records:",
    (player_team_counts < 2).sum()
)

print(
    "Matches with more than 2 team records:",
    (player_team_counts > 2).sum()
)

# Display matches without player data
print("\n3. MATCHES WITHOUT PLAYER DATA")

unmatched_matches = match_integration[
    match_integration["integration_key"].isin(
        unmatched_match_keys
    )
]

display(
    unmatched_matches[
        [
            "match_date",
            "team_1",
            "team_2",
            "source",
            "match_id",
            "result_type"
        ]
    ]
)

# Final coverage assessment
print("\n4. COVERAGE ASSESSMENT")

coverage_percentage = (
    len(matched_keys) / len(match_keys) * 100
)

print("Player data coverage:", round(coverage_percentage, 2), "%")

print("\nMATCH-KEY COVERAGE AUDIT COMPLETED")

CRICKETIQ — MATCH-KEY COVERAGE VALIDATION

1. MATCH KEY COVERAGE
Total match dataset keys: 2753
Total player dataset keys: 2748
Matching keys: 2748
Matches without player data: 5
Player keys absent from match dataset: 0

2. PLAYER TEAM-RECORD VALIDATION
Matches with exactly 2 team records: 2748
Matches with fewer than 2 team records: 0
Matches with more than 2 team records: 0

3. MATCHES WITHOUT PLAYER DATA


,match_date,team_1,team_2,source,match_id,result_type
1044,2012-07-03,Afghanistan,Ireland,Crickrida,NaN,No result
1493,2016-07-10,Afghanistan,Ireland,Crickrida,NaN,No result
1621,2017-06-14,Afghanistan,West Indies,Crickrida,NaN,No result
2448,2024-03-09,Afghanistan,Ireland,Crickrida,NaN,No result
2703,2026-08-05,Afghanistan,Ireland,Crickrida,NaN,No result



4. COVERAGE ASSESSMENT
Player data coverage: 99.82 %

MATCH-KEY COVERAGE AUDIT COMPLETED


In [20]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 8A: RESHAPE TEAM FEATURES TO MATCH LEVEL
# ============================================================

print("=" * 75)
print("CRICKETIQ — TEAM-TO-MATCH FEATURE RESHAPING")
print("=" * 75)

# Identify the aggregated feature columns
metadata_columns = [
    "match_key",
    "integration_key",
    "match_date",
    "team",
    "opponent",
    "source"
]

team_feature_columns = [
    col for col in team_integration.columns
    if col not in metadata_columns
]

# Keep team-level features and their team identity
reshape_data = team_integration[
    ["integration_key", "team"] + team_feature_columns
].copy()

# Join each team record to Notebook 05's actual team order
team_order_data = match_integration[
    ["integration_key", "team_1", "team_2"]
].copy()

reshape_data = reshape_data.merge(
    team_order_data,
    on="integration_key",
    how="left",
    validate="many_to_one"
)

# Confirm each team belongs to one side of the match
reshape_data["team_side"] = np.select(
    [
        normalize_text(reshape_data["team"])
        == normalize_text(reshape_data["team_1"]),

        normalize_text(reshape_data["team"])
        == normalize_text(reshape_data["team_2"])
    ],
    ["team_1", "team_2"],
    default="unmatched"
)

print("\nTEAM SIDE VALIDATION")
print(reshape_data["team_side"].value_counts(dropna=False))

# Identify unexpected team mappings
unmatched_teams = reshape_data[
    reshape_data["team_side"] == "unmatched"
]

print("\nUnmatched team records:", len(unmatched_teams))

# Create separate team feature tables
team_1_features = reshape_data[
    reshape_data["team_side"] == "team_1"
].copy()

team_2_features = reshape_data[
    reshape_data["team_side"] == "team_2"
].copy()

# Prefix feature names
team_1_features = team_1_features[
    ["integration_key"] + team_feature_columns
].rename(
    columns={
        col: f"team_1_player_{col}"
        for col in team_feature_columns
    }
)

team_2_features = team_2_features[
    ["integration_key"] + team_feature_columns
].rename(
    columns={
        col: f"team_2_player_{col}"
        for col in team_feature_columns
    }
)

# Combine both sides
match_player_features = team_1_features.merge(
    team_2_features,
    on="integration_key",
    how="outer",
    validate="one_to_one"
)

print("\nRESHAPED MATCH-LEVEL PLAYER DATASET")
print("Shape:", match_player_features.shape)

print("\nTeam 1 records:", len(team_1_features))
print("Team 2 records:", len(team_2_features))

print("\nMissing Team 1 feature records:",
      match_player_features.filter(
          like="team_1_player_"
      ).isna().all(axis=1).sum())

print("Missing Team 2 feature records:",
      match_player_features.filter(
          like="team_2_player_"
      ).isna().all(axis=1).sum())

print("\nSAMPLE RESHAPED RECORDS")
display(match_player_features.head())

print("\nSTEP 8A COMPLETED")

CRICKETIQ — TEAM-TO-MATCH FEATURE RESHAPING

TEAM SIDE VALIDATION
team_side
team_1    2748
team_2    2748
Name: count, dtype: int64

Unmatched team records: 0

RESHAPED MATCH-LEVEL PLAYER DATASET
Shape: (2748, 253)

Team 1 records: 2748
Team 2 records: 2748

Missing Team 1 feature records: 0
Missing Team 2 feature records: 0

SAMPLE RESHAPED RECORDS


,integration_key,team_1_player_pre_match_career_batting_average_mean,team_1_player_pre_match_career_batting_average_max,team_1_player_pre_match_career_batting_strike_rate_mean,team_1_player_pre_match_career_batting_strike_rate_max,team_1_player_pre_match_career_boundary_percentage_mean,team_1_player_pre_match_career_boundary_percentage_max,team_1_player_pre_match_career_bowling_average_mean,team_1_player_pre_match_career_bowling_average_max,team_1_player_pre_match_career_bowling_economy_mean,...,team_2_player_pre_match_recent_10_bowling_legal_balls_sum,team_2_player_pre_match_recent_10_bowling_wickets_mean,team_2_player_pre_match_recent_10_bowling_wickets_max,team_2_player_pre_match_recent_10_bowling_wickets_sum,team_2_player_recorded_players,team_2_player_player_records,team_2_player_has_player_data,team_2_player_low_player_coverage,team_2_player_full_player_coverage,team_2_player_player_coverage_ratio
0,2002-06-27|cricsheet|england|sri lanka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.000000,0.0,0.0,11,11,True,False,True,1.000000
1,2002-06-29|cricsheet|england|india,36.333333,83.0,126.827598,333.333333,46.758914,80.000000,25.208333,49.0,4.983333,...,0.0,0.000000,0.0,0.0,10,10,True,False,False,0.909091
2,2002-12-29|cricsheet|india|new zealand,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,192.0,0.454545,3.0,5.0,11,11,True,False,True,1.000000
3,2003-01-01|cricsheet|india|new zealand,34.200000,91.0,74.617281,97.282609,47.086234,100.000000,36.216667,106.0,5.023237,...,214.0,0.700000,3.0,7.0,10,10,True,False,False,0.909091
4,2003-01-04|cricsheet|india|new zealand,24.407407,62.0,57.400802,91.176471,34.501979,66.666667,21.740000,27.0,4.806788,...,413.0,1.300000,4.0,13.0,10,10,True,False,False,0.909091



STEP 8A COMPLETED


In [21]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 8B: TEAM ORDER AND FEATURE INTEGRITY AUDIT
# ============================================================

print("=" * 75)
print("CRICKETIQ — TEAM ORDER VALIDATION")
print("=" * 75)

# Verify team 1 and team 2 records
team_1_check = team_1_features.merge(
    match_integration[
        ["integration_key", "team_1"]
    ],
    on="integration_key",
    how="left",
    validate="one_to_one"
)

team_2_check = team_2_features.merge(
    match_integration[
        ["integration_key", "team_2"]
    ],
    on="integration_key",
    how="left",
    validate="one_to_one"
)

# Verify that the source team records correspond to the correct side
team_1_correct = (
    normalize_text(
        reshape_data.loc[
            reshape_data["team_side"] == "team_1", "team"
        ].reset_index(drop=True)
    )
    ==
    normalize_text(
        reshape_data.loc[
            reshape_data["team_side"] == "team_1", "team_1"
        ].reset_index(drop=True)
    )
).all()

team_2_correct = (
    normalize_text(
        reshape_data.loc[
            reshape_data["team_side"] == "team_2", "team"
        ].reset_index(drop=True)
    )
    ==
    normalize_text(
        reshape_data.loc[
            reshape_data["team_side"] == "team_2", "team_2"
        ].reset_index(drop=True)
    )
).all()

print("\nTEAM ORDER CHECK")
print("Team 1 mapping correct:", team_1_correct)
print("Team 2 mapping correct:", team_2_correct)

print("\nFEATURE COUNT CHECK")
print("Team 1 feature columns:",
      len([c for c in match_player_features.columns
           if c.startswith("team_1_player_")]))

print("Team 2 feature columns:",
      len([c for c in match_player_features.columns
           if c.startswith("team_2_player_")]))

print("\nINTEGRATION KEY CHECK")
print("Duplicate reshaped keys:",
      match_player_features["integration_key"].duplicated().sum())

print("\nSTEP 8B COMPLETED")

CRICKETIQ — TEAM ORDER VALIDATION

TEAM ORDER CHECK
Team 1 mapping correct: True
Team 2 mapping correct: True

FEATURE COUNT CHECK
Team 1 feature columns: 126
Team 2 feature columns: 126

INTEGRATION KEY CHECK
Duplicate reshaped keys: 0

STEP 8B COMPLETED


In [22]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 9A: INTEGRATE MATCH AND PLAYER FEATURES
# ============================================================

print("=" * 75)
print("CRICKETIQ — MATCH + PLAYER FEATURE INTEGRATION")
print("=" * 75)

# Left merge: preserve every Notebook 05 match
integrated_features = match_integration.merge(
    match_player_features,
    on="integration_key",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Inspect merge results
print("\nMERGE SUMMARY")
print("Integrated shape:", integrated_features.shape)

print("\nMATCHING STATUS")
print(
    integrated_features["_merge"]
    .value_counts()
)

# Player data availability
integrated_features["has_player_features"] = (
    integrated_features["_merge"] == "both"
)

print("\nPLAYER FEATURE AVAILABILITY")
print(
    integrated_features["has_player_features"]
    .value_counts()
)

# Missing player records
missing_player_matches = integrated_features[
    ~integrated_features["has_player_features"]
]

print("\nMATCHES WITHOUT PLAYER FEATURES")
display(
    missing_player_matches[
        [
            "match_date",
            "team_1",
            "team_2",
            "source",
            "result_type"
        ]
    ]
)

# Remove merge indicator after inspection
integrated_features.drop(
    columns=["_merge"],
    inplace=True
)

print("\nINTEGRATION COMPLETED")

CRICKETIQ — MATCH + PLAYER FEATURE INTEGRATION

MERGE SUMMARY
Integrated shape: (2753, 294)

MATCHING STATUS
_merge
both          2748
left_only        5
right_only       0
Name: count, dtype: int64

PLAYER FEATURE AVAILABILITY
has_player_features
True     2748
False       5
Name: count, dtype: int64

MATCHES WITHOUT PLAYER FEATURES


,match_date,team_1,team_2,source,result_type
1044,2012-07-03,Afghanistan,Ireland,Crickrida,No result
1493,2016-07-10,Afghanistan,Ireland,Crickrida,No result
1621,2017-06-14,Afghanistan,West Indies,Crickrida,No result
2448,2024-03-09,Afghanistan,Ireland,Crickrida,No result
2703,2026-08-05,Afghanistan,Ireland,Crickrida,No result



INTEGRATION COMPLETED


In [23]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 9B: FINAL INTEGRATION INTEGRITY AUDIT
# ============================================================

print("=" * 75)
print("CRICKETIQ — FINAL INTEGRATION INTEGRITY AUDIT")
print("=" * 75)

# Original match columns
original_match_columns = [
    col for col in match_integration.columns
    if col != "integration_key"
]

# All player-derived columns
player_columns = [
    col for col in match_player_features.columns
    if col != "integration_key"
]

# 1. Dataset dimensions
print("\n1. DATASET DIMENSIONS")
print("Original match rows:", len(match_integration))
print("Integrated rows:", len(integrated_features))
print("Original match columns:", len(original_match_columns))
print("Player-derived columns:", len(player_columns))
print("Total integrated columns:", len(integrated_features.columns))

# 2. Duplicate match check
print("\n2. DUPLICATE CHECK")
print(
    "Duplicate integration keys:",
    integrated_features["integration_key"].duplicated().sum()
)

# 3. Original columns preserved
missing_original_columns = [
    col for col in original_match_columns
    if col not in integrated_features.columns
]

print("\n3. ORIGINAL MATCH DATA PRESERVATION")
print("Missing original columns:", len(missing_original_columns))
print("Missing column names:", missing_original_columns)

# 4. Player feature preservation
missing_player_columns = [
    col for col in player_columns
    if col not in integrated_features.columns
]

print("\n4. PLAYER FEATURE PRESERVATION")
print("Expected player columns:", len(player_columns))
print("Missing player columns:", len(missing_player_columns))

# 5. Player availability validation
print("\n5. PLAYER DATA AVAILABILITY")

print(
    "Matches with player features:",
    integrated_features["has_player_features"].sum()
)

print(
    "Matches without player features:",
    (~integrated_features["has_player_features"]).sum()
)

# 6. Verify missing player features for unmatched matches
no_player_mask = ~integrated_features["has_player_features"]

all_player_features_missing = (
    integrated_features.loc[no_player_mask, player_columns]
    .isna()
    .all(axis=1)
)

print("\n6. UNMATCHED PLAYER FEATURE CHECK")
print(
    "Unmatched matches with all player features missing:",
    all_player_features_missing.sum()
)

# 7. Final checks
print("\n7. FINAL VALIDATION")

checks = {
    "All original match rows preserved":
        len(integrated_features) == len(match_integration),

    "No duplicate match keys":
        integrated_features["integration_key"].is_unique,

    "All original match columns preserved":
        len(missing_original_columns) == 0,

    "All player columns preserved":
        len(missing_player_columns) == 0,

    "Correct player feature availability":
        integrated_features["has_player_features"].sum() == 2748,

    "All five unmatched matches retain missing player features":
        all_player_features_missing.all()
}

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("\nALL CHECKS PASSED:",
      all(checks.values()))

print("\nINTEGRATION AUDIT COMPLETED")

CRICKETIQ — FINAL INTEGRATION INTEGRITY AUDIT

1. DATASET DIMENSIONS
Original match rows: 2753
Integrated rows: 2753
Original match columns: 40
Player-derived columns: 252
Total integrated columns: 294

2. DUPLICATE CHECK
Duplicate integration keys: 0

3. ORIGINAL MATCH DATA PRESERVATION
Missing original columns: 0
Missing column names: []

4. PLAYER FEATURE PRESERVATION
Expected player columns: 252
Missing player columns: 0

5. PLAYER DATA AVAILABILITY
Matches with player features: 2748
Matches without player features: 5

6. UNMATCHED PLAYER FEATURE CHECK
Unmatched matches with all player features missing: 5

7. FINAL VALIDATION
All original match rows preserved: PASS
No duplicate match keys: PASS
All original match columns preserved: PASS
All player columns preserved: PASS
Correct player feature availability: PASS
All five unmatched matches retain missing player features: PASS

ALL CHECKS PASSED: True

INTEGRATION AUDIT COMPLETED


In [24]:
# ============================================================
# CRICKETIQ — NOTEBOOK 08
# STEP 10: SAVE FINAL INTEGRATED DATASET
# ============================================================

from pathlib import Path
import pandas as pd

output_dir = Path(r"C:\CricIQ\data\features")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    output_dir / "cricketiq_integrated_match_features.csv"
)

# Save
integrated_features.to_csv(output_path, index=False)

# Reload and verify
published_dataset = pd.read_csv(
    output_path,
    low_memory=False
)

print("=" * 75)
print("CRICKETIQ — FINAL DATASET PUBLICATION")
print("=" * 75)

print("\nSaved path:")
print(output_path)

print("\nOriginal shape:", integrated_features.shape)
print("Published shape:", published_dataset.shape)

print(
    "\nDuplicate match keys:",
    published_dataset["integration_key"].duplicated().sum()
)

print(
    "Matches with player features:",
    published_dataset["has_player_features"].sum()
)

print(
    "Matches without player features:",
    (~published_dataset["has_player_features"]).sum()
)

if (
    published_dataset.shape == integrated_features.shape
    and published_dataset["integration_key"].is_unique
    and published_dataset["has_player_features"].sum() == 2748
):
    print("\nFINAL INTEGRATED DATASET PUBLISHED SUCCESSFULLY")
else:
    print("\nPUBLICATION VALIDATION FAILED — INVESTIGATE")

CRICKETIQ — FINAL DATASET PUBLICATION

Saved path:
C:\CricIQ\data\features\cricketiq_integrated_match_features.csv

Original shape: (2753, 294)
Published shape: (2753, 294)

Duplicate match keys: 0
Matches with player features: 2748
Matches without player features: 5

FINAL INTEGRATED DATASET PUBLISHED SUCCESSFULLY
